# Differential Gene Expression Analysis

-DATASET : GSE147334 | NCBI GEO

-STUDY   : TEX10 Knockdown vs Negative Control (shNC)

-CELL LINE: HCT116 (Human Colorectal Cancer)

-PLATFORM : RNA-seq | DESeq2 via GEO2R

-CUTOFFS  : padj ≤ 0.05 | |log2FC| ≥ 1

#  WORKFLOW :
1. Upload GEO2R results table
2. Filter significant DEGs
3. Separate Upregulated / Downregulated genes
4. Export filtered CSVs + formatted Excel
5. Generate DAVID-ready gene lists for KEGG enrichment

OUTPUT   : Filtered DEG table ready for volcano plot & KEGG


In [1]:
!pip install pandas openpyxl -q
print("✅ Libraries ready")

from google.colab import files
import io, pandas as pd

print("👇 Click 'Choose Files' below and upload your GEO2R results file")
uploaded = files.upload()

# Auto-detect filename
filename = list(uploaded.keys())[0]
print(f"\n✅ File uploaded: {filename}")

# Auto-detect separator (tab or comma)
raw = uploaded[filename].decode("utf-8")
sep = "\t" if raw.count("\t") > raw.count(",") else ","
df_raw = pd.read_csv(io.StringIO(raw), sep=sep)
df_raw.columns = df_raw.columns.str.strip()

print(f"📊 Total rows loaded : {len(df_raw):,}")
print(f"📋 Columns detected  : {list(df_raw.columns)}")

print("=== First 5 rows of raw data ===")
df_raw.head()

PADJ_CUTOFF = 0.05
LFC_CUTOFF  = 1.0       # |log2FoldChange| >= 1 means 2-fold change

# Auto-detect column names
lfc_col  = next((c for c in df_raw.columns if 'log2' in c.lower() or 'logfc' in c.lower()), None)
padj_col = next((c for c in df_raw.columns if 'padj' in c.lower()), None)
sym_col  = next((c for c in df_raw.columns if 'symbol' in c.lower()), None)

if not lfc_col:
    raise ValueError("❌ Could not find log2FoldChange column. Check column names above.")
if not padj_col:
    raise ValueError("❌ Could not find padj column. Check column names above.")

print(f"✅ Using columns → LFC: '{lfc_col}' | padj: '{padj_col}' | Symbol: '{sym_col}'")

# Work on a clean copy
df = df_raw.copy()

# Convert to numeric (in case values are strings)
df[lfc_col]  = pd.to_numeric(df[lfc_col],  errors="coerce")
df[padj_col] = pd.to_numeric(df[padj_col], errors="coerce")

# Add Direction column
def assign_direction(lfc):
    if pd.isna(lfc):        return "NA"
    if lfc >=  LFC_CUTOFF:  return "Upregulated"
    if lfc <= -LFC_CUTOFF:  return "Downregulated"
    return "NS"

df["Direction"] = df[lfc_col].apply(assign_direction)

# Add Significant flag
df["Significant"] = ((df[padj_col] <= PADJ_CUTOFF) & (df[lfc_col].abs() >= LFC_CUTOFF))

# Filter subsets
sig_all  = df[(df[padj_col] <= PADJ_CUTOFF) & (df[lfc_col].abs() >= LFC_CUTOFF)].copy()
up_genes = sig_all[sig_all["Direction"] == "Upregulated"].sort_values(lfc_col, ascending=False).reset_index(drop=True)
dn_genes = sig_all[sig_all["Direction"] == "Downregulated"].sort_values(lfc_col, ascending=True).reset_index(drop=True)
ns_genes = df[~df["Significant"]].copy()

print(f"\n{'='*45}")
print(f"  FILTER RESULTS")
print(f"  padj ≤ {PADJ_CUTOFF}  |  |log2FC| ≥ {LFC_CUTOFF}")
print(f"{'='*45}")
print(f"  Total genes tested   : {len(df):>8,}")
print(f"  Significant DEGs     : {len(sig_all):>8,}")
print(f"  ↑ Upregulated        : {len(up_genes):>8,}")
print(f"  ↓ Downregulated      : {len(dn_genes):>8,}")
print(f"  — Not significant    : {len(ns_genes):>8,}")
print(f"{'='*45}")

print("=== Top 10 Upregulated Genes ===")
display(up_genes.head(10))

print("\n=== Top 10 Downregulated Genes ===")
display(dn_genes.head(10))

import os
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT_EXCEL = "GSE147334_DEG_Filtered_Results.xlsx"
OUT_ALL      = "GSE147334_ALL_significant_DEGs.csv"
OUT_UP       = "GSE147334_Upregulated_DEGs.csv"
OUT_DOWN     = "GSE147334_Downregulated_DEGs.csv"
OUT_FULL     = "GSE147334_FullTable_with_Direction.csv"
OUT_DAVID_UP = "DAVID_upload_Upregulated_symbols.txt"
OUT_DAVID_DN = "DAVID_upload_Downregulated_symbols.txt"

# Save CSVs
sig_all.to_csv(OUT_ALL,   index=False)
up_genes.to_csv(OUT_UP,   index=False)
dn_genes.to_csv(OUT_DOWN, index=False)
df.to_csv(OUT_FULL,       index=False)

# Save DAVID gene lists (just gene symbols, one per line)
if sym_col:
    up_genes[sym_col].dropna().to_csv(OUT_DAVID_UP, index=False, header=False)
    dn_genes[sym_col].dropna().to_csv(OUT_DAVID_DN, index=False, header=False)
    print(f"✅ DAVID gene lists saved")

# ── Build styled Excel workbook ──────────────────────────────
with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    sig_all.to_excel(writer,  sheet_name="All_Significant_DEGs", index=False)
    up_genes.to_excel(writer, sheet_name="Upregulated",          index=False)
    dn_genes.to_excel(writer, sheet_name="Downregulated",        index=False)
    df.to_excel(writer,       sheet_name="Full_Table",           index=False)
    # DAVID sheet
    david_df = pd.DataFrame({
        "Upregulated_Symbols"  : pd.Series(up_genes[sym_col].dropna().values if sym_col else []),
        "Downregulated_Symbols": pd.Series(dn_genes[sym_col].dropna().values if sym_col else []),
    })
    david_df.to_excel(writer, sheet_name="DAVID_Gene_Lists", index=False)

# ── Apply Excel formatting ────────────────────────────────────
wb  = load_workbook(OUTPUT_EXCEL)
thin = Side(style="thin", color="CCCCCC")
bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)

COLORS = {
    "All_Significant_DEGs": ("1F497D", "DEEAF1"),
    "Upregulated"         : ("C00000", "FCE4D6"),
    "Downregulated"       : ("2E75B6", "D9E8F5"),
    "Full_Table"          : ("404040", "F5F5F5"),
    "DAVID_Gene_Lists"    : ("375623", "E2EFDA"),
}

def fmt_sheet(ws, hdr_hex, alt_hex):
    WHITE = "FFFFFF"
    for cell in ws[1]:
        cell.font      = Font(name="Arial", bold=True, color=WHITE, size=10)
        cell.fill      = PatternFill("solid", fgColor=hdr_hex)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = bdr
    ws.row_dimensions[1].height = 28

    for i, row in enumerate(ws.iter_rows(min_row=2), 2):
        bg = alt_hex if i % 2 == 0 else WHITE
        for cell in row:
            cell.font      = Font(name="Calibri", size=10)
            cell.fill      = PatternFill("solid", fgColor=bg)
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border    = bdr
            # Color Direction column text
            if str(ws.cell(1, cell.column).value) == "Direction":
                if cell.value == "Upregulated":
                    cell.font = Font(name="Calibri", size=10, bold=True, color="C00000")
                elif cell.value == "Downregulated":
                    cell.font = Font(name="Calibri", size=10, bold=True, color="1F497D")
                elif cell.value == "NS":
                    cell.font = Font(name="Calibri", size=10, color="888888")
            # Color Significant column
            if str(ws.cell(1, cell.column).value) == "Significant":
                if cell.value == True:
                    cell.font = Font(name="Calibri", size=10, bold=True, color="375623")
                else:
                    cell.font = Font(name="Calibri", size=10, color="888888")

    # Column widths
    for col in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(w + 4, 28)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

for sname, (h, a) in COLORS.items():
    if sname in wb.sheetnames:
        fmt_sheet(wb[sname], h, a)

# Add DAVID instructions
ws_d = wb["DAVID_Gene_Lists"]
note_row = ws_d.max_row + 2
ws_d.cell(note_row, 1,
    "→ Go to https://david.ncifcrf.gov → Paste gene list → Select 'Official Gene Symbol' → Species: Homo sapiens → Submit"
).font = Font(name="Arial", italic=True, size=9, color="595959")

wb.save(OUTPUT_EXCEL)
print(f"\n✅ All files created successfully!\n")
print(f"  📁 {OUTPUT_EXCEL}        ← Main Excel (5 sheets, formatted)")
print(f"  📄 {OUT_ALL}   ← All significant DEGs")
print(f"  📄 {OUT_UP}      ← Upregulated genes only")
print(f"  📄 {OUT_DOWN}    ← Downregulated genes only")
print(f"  📄 {OUT_FULL}    ← Full table + Direction column")
print(f"  📄 {OUT_DAVID_UP}  ← Paste into DAVID (Up)")
print(f"  📄 {OUT_DAVID_DN}  ← Paste into DAVID (Down)")


from google.colab import files

print("⬇️  Downloading all output files to your computer...\n")

for f in [OUTPUT_EXCEL, OUT_ALL, OUT_UP, OUT_DOWN, OUT_FULL, OUT_DAVID_UP, OUT_DAVID_DN]:
    if os.path.exists(f):
        files.download(f)
        print(f"  ✅ Downloaded: {f}")
    else:
        print(f"  ⚠️  Not found: {f}")

print("\n🎉 Done! All files are in your Downloads folder.")
print("Next step → Open DAVID_Gene_Lists sheet and paste symbols into DAVID for KEGG enrichment.")

✅ Libraries ready
👇 Click 'Choose Files' below and upload your GEO2R results file


Saving GSE147334.top.table (2).tsv to GSE147334.top.table (2).tsv

✅ File uploaded: GSE147334.top.table (2).tsv
📊 Total rows loaded : 18,903
📋 Columns detected  : ['GeneID', 'padj', 'pvalue', 'lfcSE', 'stat', 'log2FoldChange', 'baseMean', 'Symbol', 'Description']
=== First 5 rows of raw data ===
✅ Using columns → LFC: 'log2FoldChange' | padj: 'padj' | Symbol: 'Symbol'

  FILTER RESULTS
  padj ≤ 0.05  |  |log2FC| ≥ 1.0
  Total genes tested   :   18,903
  Significant DEGs     :      235
  ↑ Upregulated        :      103
  ↓ Downregulated      :      132
  — Not significant    :   18,668
=== Top 10 Upregulated Genes ===


,GeneID,padj,pvalue,lfcSE,stat,log2FoldChange,baseMean,Symbol,Description,Direction,Significant
0,388697,5.480000e-17,3.220000e-19,0.3480,8.960909,3.118673,70.61,HRNR,hornerin,Upregulated,True
1,2312,9.260000e-08,3.200000e-09,0.4898,5.921001,2.899951,30.17,FLG,filaggrin,Upregulated,True
2,2697,2.780000e-03,4.190000e-04,0.7102,3.528037,2.505786,74.30,GJA1,gap junction protein alpha 1,Upregulated,True
3,51286,3.650000e-02,9.730000e-03,0.9619,2.585232,2.486843,55.11,CEND1,cell cycle exit and neuronal differentiation 1,Upregulated,True
4,6326,1.220000e-04,1.090000e-05,0.4870,4.397770,2.141617,39.66,SCN2A,sodium voltage-gated channel alpha subunit 2,Upregulated,True
5,26154,3.820000e-10,7.910000e-12,0.3004,6.840241,2.054682,135.36,ABCA12,ATP binding cassette subfamily A member 12,Upregulated,True
6,3886,1.050000e-17,5.720000e-20,0.2222,9.149408,2.032677,40.38,KRT35,keratin 35,Upregulated,True
7,4608,9.120000e-09,2.480000e-10,0.3174,6.328183,2.008561,22.95,MYBPH,myosin binding protein H,Upregulated,True
8,283463,1.950000e-06,9.540000e-08,0.3615,5.335362,1.928956,117.95,MUC19,"mucin 19, oligomeric",Upregulated,True
9,8689,6.300000e-04,7.210000e-05,0.4623,3.969291,1.834962,20.57,KRT36,keratin 36,Upregulated,True



=== Top 10 Downregulated Genes ===


,GeneID,padj,pvalue,lfcSE,stat,log2FoldChange,baseMean,Symbol,Description,Direction,Significant
0,3040,9.500000e-11,1.760000e-12,0.2979,-7.052344,-2.100648,25.00,HBA2,hemoglobin subunit alpha 2,Downregulated,True
1,3039,1.400000e-09,3.180000e-11,0.2745,-6.637891,-1.822177,29.52,HBA1,hemoglobin subunit alpha 1,Downregulated,True
2,100131551,4.170000e-04,4.430000e-05,0.4410,-4.083633,-1.800980,25.95,LINC00887,long intergenic non-protein coding RNA 887,Downregulated,True
3,401562,2.870000e-05,2.100000e-06,0.3762,-4.743859,-1.784431,15.97,LCNL1,lipocalin like 1,Downregulated,True
4,79625,3.200000e-04,3.240000e-05,0.4283,-4.156150,-1.780049,16.63,NDNF,neuron derived neurotrophic factor,Downregulated,True
5,105377205,2.680000e-04,2.660000e-05,0.4227,-4.201046,-1.775796,9.49,LOC105377205,uncharacterized LOC105377205,Downregulated,True
6,104355220,4.910000e-09,1.270000e-10,0.2701,-6.430873,-1.737022,23.74,LINC01219,long intergenic non-protein coding RNA 1219,Downregulated,True
7,1958,1.140000e-12,1.350000e-14,0.2218,-7.700876,-1.708357,5763.12,EGR1,early growth response 1,Downregulated,True
8,101929174,2.000000e-08,5.820000e-10,0.2606,-6.195279,-1.614794,23.99,LOC101929174,uncharacterized LOC101929174,Downregulated,True
9,107987375,1.690000e-08,4.820000e-10,0.2591,-6.224748,-1.613077,28.24,LOC107987375,replaced by ID 104355220,Downregulated,True


...

<IPython.core.display.Javascript object>

  ✅ Downloaded: DAVID_upload_Upregulated_symbols.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✅ Downloaded: DAVID_upload_Downregulated_symbols.txt

🎉 Done! All files are in your Downloads folder.
Next step → Open DAVID_Gene_Lists sheet and paste symbols into DAVID for KEGG enrichment.
